In [2]:
from __future__ import print_function, division
from optparse import OptionParser
from collections import defaultdict, OrderedDict
import concurrent.futures
import sys
import os
import rhalphalib as rl
import numpy as np
import pickle
import gzip
import json
from coffea import hist, processor
from coffea.util import load, save
from scipy import stats
import ROOT

rl.util.install_roofit_helpers()
rl.ParametericSample.PreferRooParametricHist = False

mass_binning = [
    #0,
    40,
    50,
    60,
    70,
    80,
    90,
    100,
    120,
    150,
    180,
    240,
    300,
]
# recoil_binning=[250,310,370,470,590,840,1020,1250,3000]
#recoil_binning = [250, 310, 370, 470, 590, 3000]
recoil_binning_dict = {
    "2018": [250, 310, 370, 470, 590, 3000],
    "2017": [250, 310, 370, 470, 590, 3000],
    "2016": [250, 310, 370, 470, 590, 3000]
}

category_map = {"pass": 1, "fail": 0}

# with open("data/hf_systematic.json") as fin:
#     hf_systematic = json.load(fin)

def simple_error_propagation(pw, tw, pw2, tw2, debug=False):
    """Compute errors based on the propagation of uncertainty
    Parameters
    ----------
    pw : numpy.ndarray
        Numerator, or number of (weighted) successes, vectorized
    tw : numpy.ndarray
        Denominator or number of (weighted) trials, vectorized
    pw2 : numpy.ndarray
        Numerator sum of weights squared, vectorized
    tw2 : numpy.ndarray
        Denominator sum of weights squared, vectorized
    """
    dx = np.sqrt(pw2)
    dy = np.sqrt(tw2)
    ratio = tw / pw
    dz = ratio * np.sqrt((dx/pw)**2 + (dy/tw)**2)
    if debug:
        print('================ Propagation of uncertainty ================')
        print('ratio:', ratio)
        print('dz:', dz)
        print('================ Propagation of uncertainty ================ \n')

    return dz

def template(dictionary, process, systematic, recoil, region, category, read_sumw2=False):
    histogram = dictionary[region].integrate("process", process)
    nominal, sumw2 = histogram.integrate("systematic", "nominal").values(sumw2=True)[()]
    nominal=nominal[recoil, :, category_map[category]]
    sumw2=sumw2[recoil, :, category_map[category]]
    zerobins = nominal <= 0.
    output = nominal
    if "data" not in systematic:
        #output[zerobins] = 1.
        #sumw2[zerobins] = 0.
        if "Z+" in str(process):
            output[zerobins] = 1.
            sumw2[zerobins] = 0.
        elif "W+" in str(process) and "fail" in category:
            output[zerobins] = 1.
            sumw2[zerobins] = 0.
        elif "W+" in str(process) and "pass" in category and recoil<4:
            output[zerobins] = 1.
            sumw2[zerobins] = 0.
        elif "TT" in str(process) and "pass" in category  and recoil<4:
            output[zerobins] = 1.
            sumw2[zerobins] = 0.
        else:
            output[zerobins] = 1e-5
            sumw2[zerobins] = 0.
    if "nominal" not in systematic and "data" not in systematic:
        #print('Normalizing',systematic,'histogram of',process,'in region',region)
        output = histogram.integrate("systematic", systematic).values()[()][recoil, :, category_map[category]]
        output[zerobins] = 1.
        output[~zerobins] /= nominal[~zerobins]
        output[np.isnan(output)] = 1.
    binning = (
        dictionary[region]
        .integrate("process", process)
        .integrate("systematic", systematic)
        .axis("fjmass")
        .edges()
    )
    if read_sumw2:
        return (output, binning, "fjmass", sumw2)
    return (output, binning, "fjmass")

def remap_histograms(hists):
    data_hists = {}
    bkg_hists = {}
    signal_hists = {}

    process = hist.Cat("process", "Process", sorting="placement")
    cats = ("process",)
    sig_map = OrderedDict()
    bkg_map = OrderedDict()
    data_map = OrderedDict()
    bkg_map["Hbb"] = ("Hbb*",)
    bkg_map["DY+jets"] = ("DY+*",)
    bkg_map["VV"] = (["WW", "WZ", "ZZ"],)
    bkg_map["ST"] = ("ST*",)
    bkg_map["TT"] = ("TT*",)
    bkg_map["W+jets"] = ("W+*",)
    bkg_map["W+HF"] = ("W+HF",)
    bkg_map["W+LF"] = ("W+LF",)
    bkg_map["Z+jets"] = ("Z+*",)
    bkg_map["Z+HF"] = ("Z+HF",)
    bkg_map["Z+LF"] = ("Z+LF",)
    bkg_map["G+jets"] = ("G+*",)
    bkg_map["QCD"] = ("QCD*",)
    data_map["MET"] = ("MET",)
    data_map["SingleElectron"] = ("SingleElectron",)
    data_map["SinglePhoton"] = ("SinglePhoton",)
    data_map["EGamma"] = ("EGamma",)
    for signal in hists['sig']['template'].identifiers('process'):
        if 'mhs' not in str(signal): continue
        sig_map[str(signal)] = (str(signal),)  ## signals

    for key in hists["data"].keys():
        bkg_hists[key] = hists["bkg"][key].group(cats, process, bkg_map)
        signal_hists[key] = hists["sig"][key].group(cats, process, sig_map)
        data_hists[key] = hists["data"][key].group(cats, process, data_map)

    print('initial recoil binning',bkg_hists["template"].axis("recoil").edges())

    bkg_hists["template"] = bkg_hists["template"].rebin(
        "fjmass", hist.Bin("fjmass", "Mass", mass_binning)
    )
    signal_hists["template"] = signal_hists["template"].rebin(
        "fjmass", hist.Bin("fjmass", "Mass", mass_binning)
    )
    data_hists["template"] = data_hists["template"].rebin(
        "fjmass", hist.Bin("fjmass", "Mass", mass_binning)
    )

    bkg_hists["template"] = bkg_hists["template"].rebin(
        "recoil", hist.Bin("recoil", "Recoil", recoil_binning)
    )
    signal_hists["template"] = signal_hists["template"].rebin(
        "recoil", hist.Bin("recoil", "Recoil", recoil_binning)
    )
    data_hists["template"] = data_hists["template"].rebin(
        "recoil", hist.Bin("recoil", "Recoil", recoil_binning)
    )

    hists = {"bkg": bkg_hists, "sig": signal_hists, "data": data_hists}

    return hists

    #print(string, systUp)
def addBtagSyst(dictionary, recoil, process, region, templ, category):
    btagUp = template(dictionary, process, "btagUp", recoil, region, category)[0]
    btagDown = template(dictionary, process, "btagDown", recoil, region, category)[0]
    templ.setParamEffect(btag, btagUp, btagDown)

def addVJetsSyst(dictionary, recoil, process, region, templ, category):
    def addSyst(dictionary, recoil, process, region, templ, category, syst, string):
        histogram = dictionary[region].integrate("process", process)
        nominal=histogram.integrate("systematic", "nominal").values()[()][recoil, :, category_map[category]]
        up=histogram.integrate("systematic", string+"Up").values()[()][recoil, :, category_map[category]]
        down=histogram.integrate("systematic",string+"Down").values()[()][recoil, :, category_map[category]]
        systUp = np.array( up.sum() / nominal.sum() )
        systUp[np.isnan(systUp)] = 1.
        systUp = systUp.sum()
        templ.setParamEffect(syst, systUp)
    addSyst(dictionary, recoil, process, region, templ, category, ew1, "ew1")
    addSyst(dictionary, recoil, process, region, templ, category, ew2W, "ew2W")
    addSyst(dictionary, recoil, process, region, templ, category, ew2Z, "ew2Z")
    addSyst(dictionary, recoil, process, region, templ, category, ew3W, "ew3W")
    addSyst(dictionary, recoil, process, region, templ, category, ew3Z, "ew3Z")
    addSyst(dictionary, recoil, process, region, templ, category, mix, "mix")
    addSyst(dictionary, recoil, process, region, templ, category, qcd1, "qcd1")
    addSyst(dictionary, recoil, process, region, templ, category, qcd2, "qcd2")
    addSyst(dictionary, recoil, process, region, templ, category, qcd3, "qcd3")
'''
def addVJetsSyst(dictionary, recoil, process, region, templ, category):
    ew1Up = template(dictionary, process, "ew1Up", recoil, region, category)[0]
    ew1Down = template(dictionary, process, "ew1Down", recoil, region, category)[0]
    templ.setParamEffect(ew1, ew1Up, ew1Down)
    print("ew1",ew1Up)
    ew2GUp = template(dictionary, process, "ew2GUp", recoil, region, category)[0]
    ew2GDown = template(dictionary, process, "ew2GDown", recoil, region, category)[0]
    templ.setParamEffect(ew2G, ew2GUp, ew2GDown)

    ew2WUp = template(dictionary, process, "ew2WUp", recoil, region, category)[0]
    ew2WDown = template(dictionary, process, "ew2WDown", recoil, region, category)[0]
    templ.setParamEffect(ew2W, ew2WUp, ew2WDown)
    print("ew2W",ew2WUp)
    ew2ZUp = template(dictionary, process, "ew2ZUp", recoil, region, category)[0]
    ew2ZDown = template(dictionary, process, "ew2ZDown", recoil, region, category)[0]
    templ.setParamEffect(ew2Z, ew2ZUp, ew2ZDown)
    print("ew2Z",ew2ZUp)
    ew3GUp = template(dictionary, process, "ew3GUp", recoil, region, category)[0]
    ew3GDown = template(dictionary, process, "ew3GDown", recoil, region, category)[0]
    templ.setParamEffect(ew3G, ew3GUp, ew3GDown)
    ew3WUp = template(dictionary, process, "ew3WUp", recoil, region, category)[0]
    ew3WDown = template(dictionary, process, "ew3WDown", recoil, region, category)[0]
    templ.setParamEffect(ew3W, ew3WUp, ew3WDown)
    print("ew3W",ew3WUp)
    ew3ZUp = template(dictionary, process, "ew3ZUp", recoil, region, category)[0]
    ew3ZDown = template(dictionary, process, "ew3ZDown", recoil, region, category)[0]
    templ.setParamEffect(ew3Z, ew3ZUp, ew3ZDown)
    print("ew3Z",ew3ZUp)
    mixUp = template(dictionary, process, "mixUp", recoil, region, category)[0]
    mixDown = template(dictionary, process, "mixDown", recoil, region, category)[0]
    templ.setParamEffect(mix, mixUp, mixDown)
    print("mix",mixUp)
    muFUp = template(dictionary, process, "muFUp", recoil, region, category)[0]
    muFDown = template(dictionary, process, "muFDown", recoil, region, category)[0]
    #templ.setParamEffect(muF, muFUp, muFDown)
    muRUp = template(dictionary, process, "muRUp", recoil, region, category)[0]
    muRDown = template(dictionary, process, "muRDown", recoil, region, category)[0]
    #templ.setParamEffect(muR, muRUp, muRDown)
    qcd1Up = template(dictionary, process, "qcd1Up", recoil, region, category)[0]
    qcd1Down = template(dictionary, process, "qcd1Down", recoil, region, category)[0]
    templ.setParamEffect(qcd1, qcd1Up, qcd1Down)
    print("qcd1",qcd1Up)
    qcd2Up = template(dictionary, process, "qcd2Up", recoil, region, category)[0]
    qcd2Down = template(dictionary, process, "qcd2Down", recoil, region, category)[0]
    templ.setParamEffect(qcd2, qcd2Up, qcd2Down)
    print("qcd2",qcd2Up)
    qcd3Up = template(dictionary, process, "qcd3Up", recoil, region, category)[0]
    qcd3Down = template(dictionary, process, "qcd3Down", recoil, region, category)[0]
    templ.setParamEffect(qcd3, qcd3Up, qcd3Down)
    print("qcd3",qcd3Up)
'''
def rhalphabeth2D(process, tf_dataResidual_params, ord1, ord2):

    process_map = {
        "W+jets": 'W',
        "Z+jets": 'Z'
        }

    # Build qcd MC pass+fail model and fit to polynomial
    qcdmodel = rl.Model("qcdmodel")
    qcdpass, qcdfail = 0., 0.
    for recoilbin in range(nrecoil):
        failCh = rl.Channel(process_map[process]+"recoil%d%s" % (recoilbin, 'fail'))
        passCh = rl.Channel(process_map[process]+"recoil%d%s" % (recoilbin, 'pass'))
        qcdmodel.addChannel(failCh)
        qcdmodel.addChannel(passCh)
        # mock template
        ptnorm = 1
        #add templates
        failTempl = template(background, process, "nominal", recoilbin, "sr", "fail")
        passTempl = template(background, process, "nominal", recoilbin, "sr", "pass")
        failCh.setObservation(failTempl)
        passCh.setObservation(passTempl)
        qcdfail += failCh.getObservation().sum()
        qcdpass += passCh.getObservation().sum()

    qcdeff = qcdpass / qcdfail
    tf_MCtempl = rl.BernsteinPoly("tf_MCtempl"+process_map[process], (ord1, ord2), ['recoil', 'fjmass'], limits=(0, 10))
    tf_MCtempl_params = qcdeff * tf_MCtempl(recoilscaled, msdscaled)
    for recoilbin in range(nrecoil):
        failCh = qcdmodel[process_map[process]+'recoil%dfail' % recoilbin]
        passCh = qcdmodel[process_map[process]+'recoil%dpass' % recoilbin]
        failObs = failCh.getObservation()
        qcdparams = np.array([rl.IndependentParameter(process_map[process]+'param_recoilbin%d_msdbin%d' % (recoilbin, i), 0) for i in range(msd.nbins)])
        sigmascale = 10.
        scaledparams = failObs * (1 + sigmascale/np.maximum(1., np.sqrt(failObs)))**qcdparams
        fail_qcd = rl.ParametericSample(process_map[process]+'recoil%dfail_qcd' % recoilbin, rl.Sample.BACKGROUND, msd, scaledparams)
        failCh.addSample(fail_qcd)
        print(tf_MCtempl_params[recoilbin, :])
        pass_qcd = rl.TransferFactorSample(process_map[process]+'recoil%dpass_qcd' % recoilbin, rl.Sample.BACKGROUND, tf_MCtempl_params[recoilbin, :], fail_qcd)
        passCh.addSample(pass_qcd)

        #failCh.mask = validbins[recoilbin]
        #passCh.mask = validbins[recoilbin]

    qcdfit_ws = ROOT.RooWorkspace('qcdfit_ws')
    simpdf, obs = qcdmodel.renderRoofit(qcdfit_ws)
    qcdfit = simpdf.fitTo(obs,
                          ROOT.RooFit.Extended(True),
                          ROOT.RooFit.SumW2Error(True),
                          ROOT.RooFit.Strategy(2),
                          ROOT.RooFit.Save(),
                          ROOT.RooFit.Minimizer('Minuit2', 'migrad'),
                          ROOT.RooFit.PrintLevel(-1),
                          )
    qcdfit_ws.add(qcdfit)
    if "pytest" not in sys.modules:
         #qcdfit_ws.writeToFile(os.path.join(str(tmpdir), 'testModel_qcdfit.root'))
        qcdfit_ws.writeToFile(os.path.join(str("models"), process_map[process]+"testModel_qcdfit.root"))
    if qcdfit.status() != 0:
        raise RuntimeError('Could not fit qcd')

    param_names = [p.name for p in tf_MCtempl.parameters.reshape(-1)]
    decoVector = rl.DecorrelatedNuisanceVector.fromRooFitResult(tf_MCtempl.name + '_deco'+process_map[process], qcdfit, param_names)
    tf_MCtempl.parameters = decoVector.correlated_params.reshape(tf_MCtempl.parameters.shape)
    tf_MCtempl_params_final = tf_MCtempl(recoilscaled, msdscaled)
    tf_params = qcdeff * tf_MCtempl_params_final * tf_dataResidual_params
    return tf_params

def model(year, recoil, category, s):

    model_id = year + category + "recoil" + str(recoil)
    print(model_id)
    model = rl.Model(str(s) + model_id)

    ###
    ###
    # Signal region
    ###
    ###

    ch_name = "sr" + model_id
    sr = rl.Channel(ch_name)
    model.addChannel(sr)

    ###
    # Add data distribution to the channel
    ###

    sr.setObservation(template(data, "MET", "data", recoil, "sr", category))

    ###
    # Z(->nunu)+jets data-driven model
    ###

    if category == "pass":
        sr_zjetsMC = sr_zjetsMCPass
        sr_zjets = sr_zjetsPass
    else:
        sr_zjetsMC = sr_zjetsMCFail
        sr_zjets = sr_zjetsFail
    sr.addSample(sr_zjets)

    ###
    # W(->lnu)+jets data-driven model
    ###

    if category == "pass":
        sr_wjetsMC = sr_wjetsMCPass
        sr_wjetsTemplate = sr_wjetsMCPassTemplate
        sr_wjets = sr_wjetsPass
        if not (recoil<4):
            #sr_wjetsTemplate = template(background, "W+jets", "nominal", recoil, "sr", category, read_sumw2=True)
            sr_wjetsMC = rl.TemplateSample(
                "sr" + model_id + "_wjetsMC",
                rl.Sample.BACKGROUND,
                sr_wjetsTemplate
            )
            sr_wjetsMC.setParamEffect(lumi, 1.027)
            sr_wjetsMC.setParamEffect(trig_met, 1.02)
            sr_wjetsMC.setParamEffect(veto_tau, 1.03)
            sr_wjetsMC.setParamEffect(wjetsMC_norm, 1.4)
            sr_wjetsMC.setParamEffect(jec, 1.05)
            addBtagSyst(background, recoil, "W+jets", "sr", sr_wjetsMC, category)
            addVJetsSyst(background, recoil, "W+jets", "sr", sr_wjetsMC, category)
            sr_wjets = sr_wjetsMC
    else:
        sr_wjetsTemplate = sr_wjetsMCFailTemplate
        sr_wjetsMC = sr_wjetsMCFail
        sr_wjets = sr_wjetsFail

    sr.addSample(sr_wjets)

    ###
    # top-antitop model
    ###

    sr_ttTemplate = template(background, "TT", "nominal", recoil, "sr", category, read_sumw2=True)
    sr_ttMC = rl.TemplateSample(
        "sr" + model_id + "_ttMC",
        rl.Sample.BACKGROUND,
        sr_ttTemplate
    )
    sr_ttMC.setParamEffect(lumi, 1.027)
    sr_ttMC.setParamEffect(trig_met, 1.02)
    sr_ttMC.setParamEffect(veto_tau, 1.03)
    sr_ttMC.setParamEffect(jec, 1.05)
    addBtagSyst(background, recoil, "TT", "sr", sr_ttMC, category)

    if category == "pass" and recoil<4:
        sr_ttMC.setParamEffect(tt_norm, 1.2)
        #sr_ttMC.autoMCStats()
        sigmascale={
            '2016': 1000,
            '2017': 1000,
            '2018': 1000
        }
        sr_ttObservable = rl.Observable("fjmass", sr_ttTemplate[1])
        sr_ttParameters = np.array(
            [
                rl.IndependentParameter(
                    "sr" + year + "_tt_" + category + "_recoil"+str(recoilbin)+"_mass%d" % i,
                    0,
                    -1*sigmascale[year],
                    sigmascale[year],
                )
                for i in range(sr_ttObservable.nbins)
            ]
        )
        sr_ttBinYields = sr_ttTemplate[0] * (1 + (sigmascale[year]/np.maximum(1., np.sqrt(sr_ttTemplate[0]))))**sr_ttParameters
        #print(sr_ttBinYields)

        sr_tt = rl.ParametericSample(
            ch_name + "_tt", rl.Sample.BACKGROUND, sr_ttObservable, sr_ttBinYields
        )
        sr.addSample(sr_tt)
    else:
        sr_ttMC.setParamEffect(ttMC_norm, 1.2)
        sr.addSample(sr_ttMC)

    ###
    # Other MC-driven processes
    ###

    sr_stTemplate = template(background, "ST", "nominal", recoil, "sr", category)
    sr_st = rl.TemplateSample(ch_name + "_stMC", rl.Sample.BACKGROUND, sr_stTemplate)
    sr_st.setParamEffect(lumi, 1.027)
    sr_st.setParamEffect(trig_met, 1.02)
    sr_st.setParamEffect(veto_tau, 1.03)
    sr_st.setParamEffect(st_norm, 1.2)
    sr_st.setParamEffect(jec, 1.05)
    addBtagSyst(background, recoil, "ST", "sr", sr_st, category)
    sr.addSample(sr_st)

    sr_dyjetsTemplate = template(background, "DY+jets", "nominal", recoil, "sr", category)
    sr_dyjets = rl.TemplateSample(
        ch_name + "_dyjetsMC", rl.Sample.BACKGROUND, sr_dyjetsTemplate
    )
    sr_dyjets.setParamEffect(lumi, 1.027)
    sr_dyjets.setParamEffect(trig_met, 1.02)
    sr_dyjets.setParamEffect(veto_tau, 1.03)
    sr_dyjets.setParamEffect(zjetsMC_norm, 1.4)
    sr_dyjets.setParamEffect(jec, 1.05)
    addBtagSyst(background, recoil, "DY+jets", "sr", sr_dyjets, category)
    addVJetsSyst(background, recoil, "DY+jets", "sr", sr_dyjets, category)
    sr.addSample(sr_dyjets)

    sr_vvTemplate = template(background, "VV", "nominal", recoil, "sr", category)
    sr_vv = rl.TemplateSample(ch_name + "_vvMC", rl.Sample.BACKGROUND, sr_vvTemplate)
    sr_vv.setParamEffect(lumi, 1.027)
    sr_vv.setParamEffect(trig_met, 1.02)
    sr_vv.setParamEffect(veto_tau, 1.03)
    sr_vv.setParamEffect(vv_norm, 1.2)
    sr_vv.setParamEffect(jec, 1.05)
    addBtagSyst(background, recoil, "VV", "sr", sr_vv, category)
    sr.addSample(sr_vv)

    sr_hbbTemplate = template(background, "Hbb", "nominal", recoil, "sr", category)
    sr_hbb = rl.TemplateSample(ch_name + "_hbbMC", rl.Sample.BACKGROUND, sr_hbbTemplate)
    sr_hbb.setParamEffect(lumi, 1.027)
    sr_hbb.setParamEffect(trig_met, 1.02)
    sr_hbb.setParamEffect(veto_tau, 1.03)
    sr_hbb.setParamEffect(hbb_norm, 1.2)
    sr_hbb.setParamEffect(jec, 1.05)
    addBtagSyst(background, recoil, "Hbb", "sr", sr_hbb, category)
    sr.addSample(sr_hbb)

    sr_qcdTemplate = template(background, "QCD", "nominal", recoil, "sr", category)
    sr_qcd = rl.TemplateSample(ch_name + "_qcdMC", rl.Sample.BACKGROUND, sr_qcdTemplate)
    sr_qcd.setParamEffect(lumi, 1.027)
    sr_qcd.setParamEffect(trig_met, 1.02)
    sr_qcd.setParamEffect(veto_tau, 1.03)
    sr_qcd.setParamEffect(qcdsig_norm, 2.0)
    sr_qcd.setParamEffect(jec, 1.05)
    addBtagSyst(background, recoil, "QCD", "sr", sr_qcd, category)
    sr.addSample(sr_qcd)

    sr_signalTemplate = template(signal, s, "nominal", recoil, "sr", category)
    sr_signal = rl.TemplateSample(
        ch_name + "_" + str(s), rl.Sample.SIGNAL, sr_signalTemplate
    )
    sr_signal.setParamEffect(lumi, 1.027)
    sr_signal.setParamEffect(trig_met, 1.02)
    sr_signal.setParamEffect(veto_tau, 1.03)
    sr_signal.setParamEffect(jec, 1.05)
    addBtagSyst(signal, recoil, str(s), "sr", sr_signal, category)
    sr.addSample(sr_signal)

    ###
    # End of SR
    ###

    if category=="pass" and not (recoil<4):
        return model

    ###
    ###
    # Single muon W control region
    ###
    ###

    ch_name = "wjetm" + model_id
    wjetm = rl.Channel(ch_name)
    model.addChannel(wjetm)

    ###
    # Add data distribution to the channel
    ###

    wjetm.setObservation(template(data, "MET", "data", recoil, "wjetm", category))

    ###
    # W(->lnu)+jets data-driven model
    ###

    wjetm_wjetsTemplate = template(background, "W+jets", "nominal", recoil, "wjetm", category, read_sumw2=True)
    wjetm_wjetsMC = rl.TemplateSample(
        "wjetm" + model_id + "_wjetsMC",
        rl.Sample.BACKGROUND,
        wjetm_wjetsTemplate
    )
    wjetm_wjetsMC.setParamEffect(lumi, 1.027)
    wjetm_wjetsMC.setParamEffect(trig_met, 1.02)
    wjetm_wjetsMC.setParamEffect(veto_tau, 1.03)
    wjetm_wjetsMC.setParamEffect(wjets_norm, 1.4)
    wjetm_wjetsMC.setParamEffect(jec, 1.05)
    wjetm_wjetsMC.setParamEffect(id_mu, 1.02)
    wjetm_wjetsMC.setParamEffect(iso_mu, 1.02)
    #wjetm_wjetsMC.autoMCStats()
    addBtagSyst(background, recoil, "W+jets", "wjetm", wjetm_wjetsMC, category)
    addVJetsSyst(background, recoil, "W+jets", "wjetm", wjetm_wjetsMC, category)

    wjetm_wjetsTFstatParameters =  np.array([rl.NuisanceParameter("wjetm_"+year+"_wjetsTFstat_" + category + "_recoil"+str(recoilbin)+"_mass%d" % i, "shape") for i in range(wjetm_wjetsTemplate[0].size)])
    wjetm_wjetsTransferFactor = wjetm_wjetsMC.getExpectation() / sr_wjetsMC.getExpectation()
    nominal =  wjetm_wjetsTemplate[0] / sr_wjetsTemplate[0]
    dz = simple_error_propagation(sr_wjetsTemplate[0], wjetm_wjetsTemplate[0], sr_wjetsTemplate[3], wjetm_wjetsTemplate[3])
    wjetm_wjetsTransferFactor = wjetm_wjetsTransferFactor * ( 1. + (dz/nominal)*wjetm_wjetsTFstatParameters )
    wjetm_wjets = rl.TransferFactorSample(ch_name + "_wjets", rl.Sample.BACKGROUND, wjetm_wjetsTransferFactor, sr_wjets)
    wjetm.addSample(wjetm_wjets)

    ###
    # top-antitop model
    ###

    wjetm_ttTemplate = template(background, "TT", "nominal", recoil, "wjetm", category, read_sumw2=True)
    wjetm_ttMC = rl.TemplateSample(
        "wjetm" + model_id + "_ttMC",
        rl.Sample.BACKGROUND,
        wjetm_ttTemplate
    )
    wjetm_ttMC.setParamEffect(lumi, 1.027)
    wjetm_ttMC.setParamEffect(trig_met, 1.02)
    wjetm_ttMC.setParamEffect(veto_tau, 1.03)
    wjetm_ttMC.setParamEffect(jec, 1.05)
    wjetm_ttMC.setParamEffect(id_mu, 1.02)
    wjetm_ttMC.setParamEffect(iso_mu, 1.02)
    addBtagSyst(background, recoil, "TT", "wjetm", wjetm_ttMC, category)

    if category == "pass":
        wjetm_ttMC.setParamEffect(tt_norm, 1.2)
        #wjetm_ttMC.autoMCStats()
        wjetm_ttTFstatParameters =  np.array([rl.NuisanceParameter("wjetm_"+year+"_ttTFstat_" + category + "_recoil"+str(recoilbin)+"_mass%d" % i, "shape") for i in range(wjetm_ttTemplate[0].size)])
        wjetm_ttTransferFactor = wjetm_ttMC.getExpectation() / sr_ttMC.getExpectation()
        nominal =  wjetm_ttTemplate[0] / sr_wjetsTemplate[0]
        dz = simple_error_propagation(sr_ttTemplate[0], wjetm_ttTemplate[0], sr_ttTemplate[3], wjetm_ttTemplate[3])
        wjetm_ttTransferFactor = wjetm_ttTransferFactor * ( 1. + (dz/nominal)*wjetm_ttTFstatParameters )
        wjetm_tt = rl.TransferFactorSample(
            ch_name + "_tt", rl.Sample.BACKGROUND, wjetm_ttTransferFactor, sr_tt
        )
        wjetm.addSample(wjetm_tt)
    else:
        wjetm_ttMC.setParamEffect(ttMC_norm, 1.2)
        wjetm.addSample(wjetm_ttMC)

    ###
    # Other MC-driven processes
    ###

    wjetm_stTemplate = template(background, "ST", "nominal", recoil, "wjetm", category)
    wjetm_st = rl.TemplateSample(
        ch_name + "_stMC", rl.Sample.BACKGROUND, wjetm_stTemplate
        )
    wjetm_st.setParamEffect(lumi, 1.027)
    wjetm_st.setParamEffect(trig_met, 1.02)
    wjetm_st.setParamEffect(veto_tau, 1.03)
    wjetm_st.setParamEffect(st_norm, 1.2)
    wjetm_st.setParamEffect(jec, 1.05)
    wjetm_st.setParamEffect(id_mu, 1.02)
    wjetm_st.setParamEffect(iso_mu, 1.02)
    addBtagSyst(background, recoilbin, "ST", "wjetm", wjetm_st, category)
    wjetm.addSample(wjetm_st)

    wjetm_dyjetsTemplate = template(background, "DY+jets", "nominal", recoil, "wjetm", category)
    wjetm_dyjets = rl.TemplateSample(
        ch_name + "_dyjetsMC", rl.Sample.BACKGROUND, wjetm_dyjetsTemplate
    )
    wjetm_dyjets.setParamEffect(lumi, 1.027)
    wjetm_dyjets.setParamEffect(trig_met, 1.02)
    wjetm_dyjets.setParamEffect(veto_tau, 1.03)
    wjetm_dyjets.setParamEffect(zjetsMC_norm, 1.4)
    wjetm_dyjets.setParamEffect(jec, 1.05)
    wjetm_dyjets.setParamEffect(id_mu, 1.02)
    wjetm_dyjets.setParamEffect(iso_mu, 1.02)
    addBtagSyst(background, recoilbin, "DY+jets", "wjetm", wjetm_dyjets, category)
    addVJetsSyst(background, recoil, "DY+jets", "wjetm", wjetm_dyjets, category)
    wjetm.addSample(wjetm_dyjets)

    wjetm_vvTemplate = template(background, "VV", "nominal", recoil, "wjetm", category)
    wjetm_vv = rl.TemplateSample(
        ch_name + "_vvMC", rl.Sample.BACKGROUND, wjetm_vvTemplate
    )
    wjetm_vv.setParamEffect(lumi, 1.027)
    wjetm_vv.setParamEffect(trig_met, 1.02)
    wjetm_vv.setParamEffect(veto_tau, 1.03)
    wjetm_vv.setParamEffect(vv_norm, 1.2)
    wjetm_vv.setParamEffect(jec, 1.05)
    wjetm_vv.setParamEffect(id_mu, 1.02)
    wjetm_vv.setParamEffect(iso_mu, 1.02)
    addBtagSyst(background, recoilbin, "VV", "wjetm", wjetm_vv, category)
    wjetm.addSample(wjetm_vv)

    wjetm_hbbTemplate = template(background, "Hbb", "nominal", recoil, "wjetm", category)
    wjetm_hbb = rl.TemplateSample(
        ch_name + "_hbbMC", rl.Sample.BACKGROUND, wjetm_hbbTemplate
    )
    wjetm_hbb.setParamEffect(lumi, 1.027)
    wjetm_hbb.setParamEffect(trig_met, 1.02)
    wjetm_hbb.setParamEffect(veto_tau, 1.03)
    wjetm_hbb.setParamEffect(hbb_norm, 1.2)
    wjetm_hbb.setParamEffect(jec, 1.05)
    wjetm_hbb.setParamEffect(id_mu, 1.02)
    wjetm_hbb.setParamEffect(iso_mu, 1.02)
    addBtagSyst(background, recoilbin, "Hbb", "wjetm", wjetm_hbb, category)
    wjetm.addSample(wjetm_hbb)

    wjetm_qcdTemplate = template(background, "QCD", "nominal", recoil, "wjetm", category)
    wjetm_qcd = rl.TemplateSample(
        ch_name + "_qcdMC", rl.Sample.BACKGROUND, wjetm_qcdTemplate
    )
    wjetm_qcd.setParamEffect(lumi, 1.027)
    wjetm_qcd.setParamEffect(trig_met, 1.02)
    wjetm_qcd.setParamEffect(veto_tau, 1.03)
    wjetm_qcd.setParamEffect(qcdmu_norm, 2.0)
    wjetm_qcd.setParamEffect(jec, 1.05)
    wjetm_qcd.setParamEffect(id_mu, 1.02)
    wjetm_qcd.setParamEffect(iso_mu, 1.02)
    addBtagSyst(background, recoilbin, "QCD", "wjetm", wjetm_qcd, category)
    wjetm.addSample(wjetm_qcd)

    ###
    # End of single muon W control region
    ###

    ###
    ###
    # Single electron W control region
    ###
    ###

    ch_name = "wjete" + model_id
    wjete = rl.Channel(ch_name)
    model.addChannel(wjete)

    ###
    # Add data distribution to the channel
    ###

    if year == "2018":
        wjete.setObservation(template(data, "EGamma", "data", recoil, "wjete", category))
    else:
        wjete.setObservation(template(data, "SingleElectron", "data", recoil, "wjete", category))

    ###
    # W(->lnu)+jets data-driven model
    ###

    wjete_wjetsTemplate = template(background, "W+jets", "nominal", recoil, "wjete", category, read_sumw2=True)
    wjete_wjetsMC = rl.TemplateSample(
        "wjete" + model_id + "_wjetsMC",
        rl.Sample.BACKGROUND,
        wjete_wjetsTemplate
    )
    wjete_wjetsMC.setParamEffect(lumi, 1.027)
    wjete_wjetsMC.setParamEffect(trig_e, 1.01)
    wjete_wjetsMC.setParamEffect(veto_tau, 1.03)
    wjete_wjetsMC.setParamEffect(wjets_norm, 1.4)
    wjete_wjetsMC.setParamEffect(jec, 1.05)
    wjete_wjetsMC.setParamEffect(id_e, 1.02)
    wjete_wjetsMC.setParamEffect(reco_e, 1.02)
    #wjete_wjetsMC.autoMCStats()
    addBtagSyst(background, recoil, "W+jets", "wjete", wjete_wjetsMC, category)
    addVJetsSyst(background, recoil, "W+jets", "wjete", wjete_wjetsMC, category)

    wjete_wjetsTFstatParameters =  np.array([rl.NuisanceParameter("wjete_"+year+"_wjetsTFstat_" + category + "_recoil"+str(recoilbin)+"_mass%d" % i, "shape") for i in range(wjete_wjetsTemplate[0].size)])
    wjete_wjetsTransferFactor = wjete_wjetsMC.getExpectation() / sr_wjetsMC.getExpectation()
    nominal =  wjete_wjetsTemplate[0] / sr_wjetsTemplate[0]
    dz = simple_error_propagation(sr_wjetsTemplate[0], wjete_wjetsTemplate[0], sr_wjetsTemplate[3], wjete_wjetsTemplate[3])
    wjete_wjetsTransferFactor = wjete_wjetsTransferFactor * ( 1. + (dz/nominal)*wjete_wjetsTFstatParameters )
    wjete_wjets = rl.TransferFactorSample(
        ch_name + "_wjets", rl.Sample.BACKGROUND, wjete_wjetsTransferFactor, sr_wjets
    )
    wjete.addSample(wjete_wjets)

    ###
    # top-antitop model
    ###

    wjete_ttTemplate = template(background, "TT", "nominal", recoil, "wjete", category, read_sumw2=True)
    wjete_ttMC = rl.TemplateSample(
        "wjete" + model_id + "_ttMC",
        rl.Sample.BACKGROUND,
        wjete_ttTemplate
    )
    wjete_ttMC.setParamEffect(lumi, 1.027)
    wjete_ttMC.setParamEffect(trig_e, 1.01)
    wjete_ttMC.setParamEffect(veto_tau, 1.03)
    wjete_ttMC.setParamEffect(jec, 1.05)
    wjete_ttMC.setParamEffect(id_e, 1.02)
    wjete_ttMC.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, recoil, "TT", "wjete", wjete_ttMC, category)

    if category == "pass":
        wjete_ttMC.setParamEffect(tt_norm, 1.2)
        #wjete_ttMC.autoMCStats()
        wjete_ttTFstatParameters =  np.array([rl.NuisanceParameter("wjete_"+year+"_ttTFstat_" + category + "_recoil"+str(recoilbin)+"_mass%d" % i, "shape") for i in range(wjete_ttTemplate[0].size)])
        wjete_ttTransferFactor = wjete_ttMC.getExpectation() / sr_ttMC.getExpectation()
        nominal =  wjete_ttTemplate[0] / sr_ttTemplate[0]
        dz = simple_error_propagation(sr_ttTemplate[0], wjete_ttTemplate[0], sr_ttTemplate[3], wjete_ttTemplate[3])
        wjete_ttTransferFactor = wjete_wjetsTransferFactor * ( 1. + (dz/nominal)*wjete_ttTFstatParameters )
        wjete_tt = rl.TransferFactorSample(
            ch_name + "_tt", rl.Sample.BACKGROUND, wjete_ttTransferFactor, sr_tt
        )
        wjete.addSample(wjete_tt)
    else:
        wjete_ttMC.setParamEffect(ttMC_norm, 1.2)
        wjete.addSample(wjete_ttMC)

    ###
    # Other MC-driven processes
    ###

    wjete_stTemplate = template(background, "ST", "nominal", recoil, "wjete", category)
    wjete_st = rl.TemplateSample(
        ch_name + "_stMC", rl.Sample.BACKGROUND, wjete_stTemplate
    )
    wjete_st.setParamEffect(lumi, 1.027)
    wjete_st.setParamEffect(trig_e, 1.01)
    wjete_st.setParamEffect(veto_tau, 1.03)
    wjete_st.setParamEffect(st_norm, 1.2)
    wjete_st.setParamEffect(jec, 1.05)
    wjete_st.setParamEffect(id_e, 1.02)
    wjete_st.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, recoilbin, "ST", "wjete", wjete_st, category)
    wjete.addSample(wjete_st)

    wjete_dyjetsTemplate = template(background, "DY+jets", "nominal", recoil, "wjete", category)
    wjete_dyjets = rl.TemplateSample(
        ch_name + "_dyjetsMC", rl.Sample.BACKGROUND, wjete_dyjetsTemplate
    )
    wjete_dyjets.setParamEffect(lumi, 1.027)
    wjete_dyjets.setParamEffect(trig_e, 1.01)
    wjete_dyjets.setParamEffect(veto_tau, 1.03)
    wjete_dyjets.setParamEffect(zjetsMC_norm, 1.4)
    wjete_dyjets.setParamEffect(jec, 1.05)
    wjete_dyjets.setParamEffect(id_e, 1.02)
    wjete_dyjets.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, recoilbin, "DY+jets", "wjete", wjete_dyjets, category)
    addVJetsSyst(background, recoil, "DY+jets", "wjete", wjete_dyjets, category)
    wjete.addSample(wjete_dyjets)

    wjete_vvTemplate = template(background, "VV", "nominal", recoil, "wjete", category)
    wjete_vv = rl.TemplateSample(
        ch_name + "_vvMC", rl.Sample.BACKGROUND, wjete_vvTemplate
    )
    wjete_vv.setParamEffect(lumi, 1.027)
    wjete_vv.setParamEffect(trig_e, 1.01)
    wjete_vv.setParamEffect(veto_tau, 1.03)
    wjete_vv.setParamEffect(vv_norm, 1.2)
    wjete_vv.setParamEffect(jec, 1.05)
    wjete_vv.setParamEffect(id_e, 1.02)
    wjete_vv.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, recoilbin, "VV", "wjete", wjete_vv, category)
    wjete.addSample(wjete_vv)

    wjete_hbbTemplate = template(background, "Hbb", "nominal", recoil, "wjete", category)
    wjete_hbb = rl.TemplateSample(
        ch_name + "_hbbMC", rl.Sample.BACKGROUND, wjete_hbbTemplate
    )
    wjete_hbb.setParamEffect(lumi, 1.027)
    wjete_hbb.setParamEffect(trig_e, 1.01)
    wjete_hbb.setParamEffect(veto_tau, 1.03)
    wjete_hbb.setParamEffect(hbb_norm, 1.2)
    wjete_hbb.setParamEffect(jec, 1.05)
    wjete_hbb.setParamEffect(id_e, 1.02)
    wjete_hbb.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, recoilbin, "Hbb", "wjete", wjete_hbb, category)
    wjete.addSample(wjete_hbb)

    wjete_qcdTemplate = template(background, "QCD", "nominal", recoil, "wjete", category)
    wjete_qcd = rl.TemplateSample(
        ch_name + "_qcdMC", rl.Sample.BACKGROUND, wjete_qcdTemplate
    )
    wjete_qcd.setParamEffect(lumi, 1.027)
    wjete_qcd.setParamEffect(trig_e, 1.01)
    wjete_qcd.setParamEffect(veto_tau, 1.03)
    wjete_qcd.setParamEffect(qcde_norm, 2.0)
    wjete_qcd.setParamEffect(jec, 1.05)
    wjete_qcd.setParamEffect(id_e, 1.02)
    wjete_qcd.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, recoilbin, "QCD", "wjete", wjete_qcd, category)
    wjete.addSample(wjete_qcd)

    ###
    # End of single electron W control region
    ###

    if not (category=="pass"):
        return model

    ###
    ###
    # Single muon top control region
    ###
    ###

    ch_name = "ttbarm" + model_id
    ttbarm = rl.Channel(ch_name)
    model.addChannel(ttbarm)

    ###
    # Add data distribution to the channel
    ###

    ttbarm.setObservation(template(data, "MET", "data", recoil, "ttbarm", category))

    ###
    # top-antitop model
    ###

    ttbarm_ttTemplate = template(background, "TT", "nominal", recoil, "ttbarm", category, read_sumw2=True)
    ttbarm_ttMC = rl.TemplateSample(
        "ttbarm" + model_id + "_ttMC",
        rl.Sample.BACKGROUND,
        ttbarm_ttTemplate
    )
    ttbarm_ttMC.setParamEffect(lumi, 1.027)
    ttbarm_ttMC.setParamEffect(trig_met, 1.02)
    ttbarm_ttMC.setParamEffect(veto_tau, 1.03)
    ttbarm_ttMC.setParamEffect(tt_norm, 1.2)
    ttbarm_ttMC.setParamEffect(jec, 1.05)
    ttbarm_ttMC.setParamEffect(id_mu, 1.02)
    ttbarm_ttMC.setParamEffect(iso_mu, 1.02)
    addBtagSyst(background, recoil, "TT", "ttbarm", ttbarm_ttMC, category)
    #ttbarm_ttMC.autoMCStats()

    ttbarm_ttTFstatParameters =  np.array([rl.NuisanceParameter("ttbarm_"+year+"_ttTFstat_" + category + "_recoil"+str(recoilbin)+"_mass%d" % i, "shape") for i in range(ttbarm_ttTemplate[0].size)])
    ttbarm_ttTransferFactor = ttbarm_ttMC.getExpectation() / sr_ttMC.getExpectation()
    nominal =  ttbarm_ttTemplate[0] / sr_ttTemplate[0]
    dz = simple_error_propagation(sr_ttTemplate[0], ttbarm_ttTemplate[0], sr_ttTemplate[3], ttbarm_ttTemplate[3])
    ttbarm_ttTransferFactor = ttbarm_ttTransferFactor * ( 1. + (dz/nominal)*ttbarm_ttTFstatParameters )
    ttbarm_tt = rl.TransferFactorSample(
        ch_name + "_tt", rl.Sample.BACKGROUND, ttbarm_ttTransferFactor, sr_tt
    )
    ttbarm.addSample(ttbarm_tt)

    ###
    # Other MC-driven processes
    ###

    ttbarm_wjetsTemplate = template(background, "W+jets", "nominal", recoil, "ttbarm", category)
    ttbarm_wjets = rl.TemplateSample(
        ch_name + "_wjetsMC", rl.Sample.BACKGROUND, ttbarm_wjetsTemplate
    )
    ttbarm_wjets.setParamEffect(lumi, 1.027)
    ttbarm_wjets.setParamEffect(trig_met, 1.02)
    ttbarm_wjets.setParamEffect(veto_tau, 1.03)
    ttbarm_wjets.setParamEffect(wjetsMC_norm, 1.4)
    ttbarm_wjets.setParamEffect(jec, 1.05)
    ttbarm_wjets.setParamEffect(id_mu, 1.02)
    ttbarm_wjets.setParamEffect(iso_mu, 1.02)
    addBtagSyst(background, recoilbin, "W+jets", "ttbarm", ttbarm_wjets, category)
    addVJetsSyst(background, recoil, "W+jets", "ttbarm", ttbarm_wjets, category)
    ttbarm.addSample(ttbarm_wjets)

    ttbarm_stTemplate = template(background, "ST", "nominal", recoil, "ttbarm", category)
    ttbarm_st = rl.TemplateSample(
        ch_name + "_stMC", rl.Sample.BACKGROUND, ttbarm_stTemplate
    )
    ttbarm_st.setParamEffect(lumi, 1.027)
    ttbarm_st.setParamEffect(trig_met, 1.02)
    ttbarm_st.setParamEffect(veto_tau, 1.03)
    ttbarm_st.setParamEffect(st_norm, 1.2)
    ttbarm_st.setParamEffect(jec, 1.05)
    ttbarm_st.setParamEffect(id_mu, 1.02)
    ttbarm_st.setParamEffect(iso_mu, 1.02)
    addBtagSyst(background, recoilbin, "ST", "ttbarm", ttbarm_st, category)
    ttbarm.addSample(ttbarm_st)

    ttbarm_dyjetsTemplate = template(background, "DY+jets", "nominal", recoil, "ttbarm", category)
    ttbarm_dyjets = rl.TemplateSample(
        ch_name + "_dyjetsMC", rl.Sample.BACKGROUND, ttbarm_dyjetsTemplate
    )
    ttbarm_dyjets.setParamEffect(lumi, 1.027)
    ttbarm_dyjets.setParamEffect(trig_met, 1.02)
    ttbarm_dyjets.setParamEffect(veto_tau, 1.03)
    ttbarm_dyjets.setParamEffect(zjetsMC_norm, 1.4)
    ttbarm_dyjets.setParamEffect(jec, 1.05)
    ttbarm_dyjets.setParamEffect(id_mu, 1.02)
    ttbarm_dyjets.setParamEffect(iso_mu, 1.02)
    addBtagSyst(background, recoilbin, "DY+jets", "ttbarm", ttbarm_dyjets, category)
    addVJetsSyst(background, recoil, "DY+jets", "ttbarm", ttbarm_dyjets, category)
    ttbarm.addSample(ttbarm_dyjets)

    ttbarm_vvTemplate = template(background, "VV", "nominal", recoil, "ttbarm", category)
    ttbarm_vv = rl.TemplateSample(
        ch_name + "_vvMC", rl.Sample.BACKGROUND, ttbarm_vvTemplate
    )
    ttbarm_vv.setParamEffect(lumi, 1.027)
    ttbarm_vv.setParamEffect(trig_met, 1.02)
    ttbarm_vv.setParamEffect(veto_tau, 1.03)
    ttbarm_vv.setParamEffect(vv_norm, 1.2)
    ttbarm_vv.setParamEffect(jec, 1.05)
    ttbarm_vv.setParamEffect(id_mu, 1.02)
    ttbarm_vv.setParamEffect(iso_mu, 1.02)
    addBtagSyst(background, recoilbin, "VV", "ttbarm", ttbarm_vv, category)
    ttbarm.addSample(ttbarm_vv)

    ttbarm_hbbTemplate = template(background, "Hbb", "nominal", recoil, "ttbarm", category)
    ttbarm_hbb = rl.TemplateSample(
        ch_name + "_hbbMC", rl.Sample.BACKGROUND, ttbarm_hbbTemplate
    )
    ttbarm_hbb.setParamEffect(lumi, 1.027)
    ttbarm_hbb.setParamEffect(trig_met, 1.02)
    ttbarm_hbb.setParamEffect(veto_tau, 1.03)
    ttbarm_hbb.setParamEffect(hbb_norm, 1.2)
    ttbarm_hbb.setParamEffect(jec, 1.05)
    ttbarm_hbb.setParamEffect(id_mu, 1.02)
    ttbarm_hbb.setParamEffect(iso_mu, 1.02)
    addBtagSyst(background, recoilbin, "Hbb", "ttbarm", ttbarm_hbb, category)
    ttbarm.addSample(ttbarm_hbb)

    ttbarm_qcdTemplate = template(background, "QCD", "nominal", recoil, "ttbarm", category)
    ttbarm_qcd = rl.TemplateSample(
        ch_name + "_qcdMC", rl.Sample.BACKGROUND, ttbarm_qcdTemplate
    )
    ttbarm_qcd.setParamEffect(lumi, 1.027)
    ttbarm_qcd.setParamEffect(trig_met, 1.02)
    ttbarm_qcd.setParamEffect(veto_tau, 1.03)
    ttbarm_qcd.setParamEffect(qcdmu_norm, 2.0)
    ttbarm_qcd.setParamEffect(jec, 1.05)
    ttbarm_qcd.setParamEffect(id_mu, 1.02)
    ttbarm_qcd.setParamEffect(iso_mu, 1.02)
    addBtagSyst(background, recoilbin, "QCD", "ttbarm", ttbarm_qcd, category)
    ttbarm.addSample(ttbarm_qcd)

    ###
    # End of single muon top control region
    ###

    ###
    ###
    # Single electron top control region
    ###
    ###

    ch_name = "ttbare" + model_id
    ttbare = rl.Channel(ch_name)
    model.addChannel(ttbare)

    ###
    # Add data distribution to the channel
    ###

    if year == "2018":
        ttbare.setObservation(template(data, "EGamma", "data", recoil, "ttbare", category))
    else:
        ttbare.setObservation(template(data, "SingleElectron", "data", recoil, "ttbare", category))

    ###
    # top-antitop model
    ###

    ttbare_ttTemplate = template(background, "TT", "nominal", recoil, "ttbare", category, read_sumw2=True)
    ttbare_ttMC = rl.TemplateSample(
        "ttbare" + model_id + "_ttMC",
        rl.Sample.BACKGROUND,
        ttbare_ttTemplate
    )
    ttbare_ttMC.setParamEffect(lumi, 1.027)
    ttbare_ttMC.setParamEffect(trig_e, 1.01)
    ttbare_ttMC.setParamEffect(veto_tau, 1.03)
    ttbare_ttMC.setParamEffect(tt_norm, 1.2)
    ttbare_ttMC.setParamEffect(jec, 1.05)
    ttbare_ttMC.setParamEffect(id_e, 1.02)
    ttbare_ttMC.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, recoil, "TT", "ttbare", ttbare_ttMC, category)
    #ttbare_ttMC.autoMCStats()

    ttbare_ttTFstatParameters =  np.array([rl.NuisanceParameter("ttbare_"+year+"_ttTFstat_" + category + "_recoil"+str(recoilbin)+"_mass%d" % i, "shape") for i in range(ttbare_ttTemplate[0].size)])
    ttbare_ttTransferFactor = ttbare_ttMC.getExpectation() / sr_ttMC.getExpectation()
    nominal =  ttbare_ttTemplate[0] / sr_ttTemplate[0]
    dz = simple_error_propagation(sr_ttTemplate[0], ttbare_ttTemplate[0], sr_ttTemplate[3], ttbare_ttTemplate[3])
    ttbare_ttTransferFactor = ttbare_ttTransferFactor * ( 1. + (dz/nominal)*ttbare_ttTFstatParameters )
    ttbare_tt = rl.TransferFactorSample(
        ch_name + "_tt", rl.Sample.BACKGROUND, ttbare_ttTransferFactor, sr_tt
    )
    ttbare.addSample(ttbare_tt)

    ###
    # Other MC-driven processes
    ###

    ttbare_wjetsTemplate = template(background, "W+jets", "nominal", recoil, "ttbare", category)
    ttbare_wjets = rl.TemplateSample(
        ch_name + "_wjetsMC", rl.Sample.BACKGROUND, ttbare_wjetsTemplate
    )
    ttbare_wjets.setParamEffect(lumi, 1.027)
    ttbare_wjets.setParamEffect(trig_e, 1.01)
    ttbare_wjets.setParamEffect(veto_tau, 1.03)
    ttbare_wjets.setParamEffect(wjetsMC_norm, 1.4)
    ttbare_wjets.setParamEffect(jec, 1.05)
    ttbare_wjets.setParamEffect(id_e, 1.02)
    ttbare_wjets.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, recoilbin, "W+jets", "ttbare", ttbare_wjets, category)
    addVJetsSyst(background, recoil, "W+jets", "ttbare", ttbare_wjets, category)
    ttbare.addSample(ttbare_wjets)

    ttbare_stTemplate = template(background, "ST", "nominal", recoil, "ttbare", category)
    ttbare_st = rl.TemplateSample(
        ch_name + "_stMC", rl.Sample.BACKGROUND, ttbare_stTemplate
    )
    ttbare_st.setParamEffect(lumi, 1.027)
    ttbare_st.setParamEffect(trig_e, 1.01)
    ttbare_st.setParamEffect(veto_tau, 1.03)
    ttbare_st.setParamEffect(st_norm, 1.2)
    ttbare_st.setParamEffect(jec, 1.05)
    ttbare_st.setParamEffect(id_e, 1.02)
    ttbare_st.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, recoilbin, "ST", "ttbare", ttbare_st, category)
    ttbare.addSample(ttbare_st)

    ttbare_dyjetsTemplate = template(background, "DY+jets", "nominal", recoil, "ttbare", category)
    ttbare_dyjets = rl.TemplateSample(
        ch_name + "_dyjetsMC", rl.Sample.BACKGROUND, ttbare_dyjetsTemplate
    )
    ttbare_dyjets.setParamEffect(lumi, 1.027)
    ttbare_dyjets.setParamEffect(trig_e, 1.01)
    ttbare_dyjets.setParamEffect(veto_tau, 1.03)
    ttbare_dyjets.setParamEffect(zjetsMC_norm, 1.4)
    ttbare_dyjets.setParamEffect(jec, 1.05)
    ttbare_dyjets.setParamEffect(id_e, 1.02)
    ttbare_dyjets.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, recoilbin, "DY+jets", "ttbare", ttbare_dyjets, category)
    addVJetsSyst(background, recoil, "DY+jets", "ttbare", ttbare_dyjets, category)
    ttbare.addSample(ttbare_dyjets)

    ttbare_vvTemplate = template(background, "VV", "nominal", recoil, "ttbare", category)
    ttbare_vv = rl.TemplateSample(
        ch_name + "_vvMC", rl.Sample.BACKGROUND, ttbare_vvTemplate
    )
    ttbare_vv.setParamEffect(lumi, 1.027)
    ttbare_vv.setParamEffect(trig_e, 1.01)
    ttbare_vv.setParamEffect(veto_tau, 1.03)
    ttbare_vv.setParamEffect(vv_norm, 1.2)
    ttbare_vv.setParamEffect(jec, 1.05)
    ttbare_vv.setParamEffect(id_e, 1.02)
    ttbare_vv.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, recoilbin, "VV", "ttbare", ttbare_vv, category)
    ttbare.addSample(ttbare_vv)

    ttbare_hbbTemplate = template(background, "Hbb", "nominal", recoil, "ttbare", category)
    ttbare_hbb = rl.TemplateSample(
        ch_name + "_hbbMC", rl.Sample.BACKGROUND, ttbare_hbbTemplate
    )
    ttbare_hbb.setParamEffect(lumi, 1.027)
    ttbare_hbb.setParamEffect(trig_e, 1.01)
    ttbare_hbb.setParamEffect(veto_tau, 1.03)
    ttbare_hbb.setParamEffect(hbb_norm, 1.2)
    ttbare_hbb.setParamEffect(jec, 1.05)
    ttbare_hbb.setParamEffect(id_e, 1.02)
    ttbare_hbb.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, recoilbin, "Hbb", "ttbare", ttbare_hbb, category)
    ttbare.addSample(ttbare_hbb)

    ttbare_qcdTemplate = template(background, "QCD", "nominal", recoil, "ttbare", category)
    ttbare_qcd = rl.TemplateSample(
        ch_name + "_qcdMC", rl.Sample.BACKGROUND, ttbare_qcdTemplate
    )
    ttbare_qcd.setParamEffect(lumi, 1.027)
    ttbare_qcd.setParamEffect(trig_e, 1.01)
    ttbare_qcd.setParamEffect(veto_tau, 1.03)
    ttbare_qcd.setParamEffect(qcde_norm, 2.0)
    ttbare_qcd.setParamEffect(jec, 1.05)
    ttbare_qcd.setParamEffect(id_e, 1.02)
    ttbare_qcd.setParamEffect(reco_e, 1.02)
    addBtagSyst(background, recoilbin, "QCD", "ttbare", ttbare_qcd, category)
    ttbare.addSample(ttbare_qcd)

    ###
    # End of single electron top control region
    ###

    return model


In [4]:
lumi = rl.NuisanceParameter("lumi" + '2018', "lnN")

In [22]:
recoil_binning = recoil_binning_dict['2018']
recoilbins = np.array(recoil_binning)
nrecoil = len(recoilbins) - 1
ptpts, msdpts = np.meshgrid(recoilbins[:-1] + 0.3 * np.diff(recoilbins), msdbins[:-1] + 0.5 * np.diff(msdbins), indexing='ij')
print(recoilbins)
print(ptpts)
print(msdbins)
print(msdpts)

recoilscaled = (ptpts - 250.) / (3000. - 250.)
#    msdpts = np.sqrt(msdpts) * np.sqrt(msdpts)
msdscaled = (msdpts - 40.) / (300.0 - 40.)
print(recoilscaled)
print(msdscaled)

[ 250  310  370  470  590 3000]
[[ 268.  268.  268.  268.  268.  268.  268.  268.  268.  268.  268.]
 [ 328.  328.  328.  328.  328.  328.  328.  328.  328.  328.  328.]
 [ 400.  400.  400.  400.  400.  400.  400.  400.  400.  400.  400.]
 [ 506.  506.  506.  506.  506.  506.  506.  506.  506.  506.  506.]
 [1313. 1313. 1313. 1313. 1313. 1313. 1313. 1313. 1313. 1313. 1313.]]
[ 40  50  60  70  80  90 100 120 150 180 240 300]
[[ 45.  55.  65.  75.  85.  95. 110. 135. 165. 210. 270.]
 [ 45.  55.  65.  75.  85.  95. 110. 135. 165. 210. 270.]
 [ 45.  55.  65.  75.  85.  95. 110. 135. 165. 210. 270.]
 [ 45.  55.  65.  75.  85.  95. 110. 135. 165. 210. 270.]
 [ 45.  55.  65.  75.  85.  95. 110. 135. 165. 210. 270.]]
[[0.00654545 0.00654545 0.00654545 0.00654545 0.00654545 0.00654545
  0.00654545 0.00654545 0.00654545 0.00654545 0.00654545]
 [0.02836364 0.02836364 0.02836364 0.02836364 0.02836364 0.02836364
  0.02836364 0.02836364 0.02836364 0.02836364 0.02836364]
 [0.05454545 0.05454545 0.054

In [19]:
msd.name

'fjmass'

In [ ]:
if __name__ == "__main__":
    if not os.path.exists("datacards"):
        os.mkdir("datacards")
    parser = OptionParser()
    parser.add_option("-y", "--year", help="year", dest="year", default="")
    (options, args) = parser.parse_args()
    year = options.year
    recoil_binning = recoil_binning_dict[year]
    recoilbins = np.array(recoil_binning)
    nrecoil = len(recoilbins) - 1

    ###
    # Extract histograms from input file
    ###

    print("Grouping histograms")
    hists = load("hists/darkhiggs" + year + ".scaled")
    hists = remap_histograms(hists)
    data_hists = hists["data"]
    bkg_hists = hists["bkg"]
    signal_hists = hists["sig"]

    ###
    # Preparing histograms for fit
    ##

    data = {}
    for r in data_hists["template"].identifiers("region"):
        data[str(r)] = data_hists["template"].integrate("region", r).sum("gentype")

    background = {}
    for r in bkg_hists["template"].identifiers("region"):
        background[str(r)] = bkg_hists["template"].integrate("region", r).sum("gentype")

    signal = {}
    for r in bkg_hists["template"].identifiers("region"):
        signal[str(r)] = signal_hists["template"].integrate("region", r).sum("gentype")

    ###
    ###
    # Setting up systematics
    ###
    ###
    lumi = rl.NuisanceParameter("lumi" + year, "lnN")
    zjets_norm = rl.NuisanceParameter("zjets_norm", "lnN")
    wjets_norm = rl.NuisanceParameter("wjets_norm", "lnN")
    tt_norm = rl.NuisanceParameter("tt_norm", "lnN")
    id_e = rl.NuisanceParameter("id_e" + year, "lnN")
    id_mu = rl.NuisanceParameter("id_mu" + year, "lnN")
    id_pho = rl.NuisanceParameter("id_pho" + year, "lnN")
    reco_e = rl.NuisanceParameter("reco_e" + year, "lnN")
    iso_mu = rl.NuisanceParameter("iso_mu" + year, "lnN")
    trig_e = rl.NuisanceParameter("trig_e" + year, "lnN")
    trig_met = rl.NuisanceParameter("trig_met" + year, "lnN")
    trig_pho = rl.NuisanceParameter("trig_pho" + year, "lnN")
    veto_tau = rl.NuisanceParameter("veto_tau" + year, "lnN")
    jec = rl.NuisanceParameter("jec" + year, "lnN")
    #btag = rl.NuisanceParameter("btag" + year, "shapeN")  # AK4 btag
    btag = rl.NuisanceParameter("btag" + year, "shape")  # AK4 btag
    ew1 = rl.NuisanceParameter("ew1", "lnN")
    #ew2G = rl.NuisanceParameter("ew2G", "lnN")
    ew2W = rl.NuisanceParameter("ew2W", "lnN")
    ew2Z = rl.NuisanceParameter("ew2Z", "lnN")
    #ew3G = rl.NuisanceParameter("ew3G", "lnN")
    ew3W = rl.NuisanceParameter("ew3W", "lnN")
    ew3Z = rl.NuisanceParameter("ew3Z", "lnN")
    mix = rl.NuisanceParameter("mix", "lnN")
    #muF = rl.NuisanceParameter("muF", "lnN")
    #muR = rl.NuisanceParameter("muR", "lnN")
    qcd1 = rl.NuisanceParameter("qcd1", "lnN")
    qcd2 = rl.NuisanceParameter("qcd2", "lnN")
    qcd3 = rl.NuisanceParameter("qcd3", "lnN")
    whf_fraction = rl.NuisanceParameter("whf_fraction", "lnN")
    zhf_fraction = rl.NuisanceParameter("zhf_fraction", "lnN")
    #ghf_fraction = rl.NuisanceParameter("ghf_fraction", "shapeN")

    ###
    # Preparing Rhalphabet
    ###

    msdbins = np.array(mass_binning)
    msd = rl.Observable('fjmass', msdbins)
    # here we derive these all at once with 2D array
    ptpts, msdpts = np.meshgrid(recoilbins[:-1] + 0.3 * np.diff(recoilbins), msdbins[:-1] + 0.5 * np.diff(msdbins), indexing='ij')
    print(recoilbins)
    print(ptpts)
    print(msdbins)
    print(msdpts)
    recoilscaled = (ptpts - 250.) / (3000. - 250.)
    #    msdpts = np.sqrt(msdpts) * np.sqrt(msdpts)
    msdscaled = (msdpts - 40.) / (300.0 - 40.)
    print(recoilscaled)
    print(msdscaled)

    tf_dataResidualW = rl.BernsteinPoly("tf_dataResidualW"+year, (1, 1), ['recoil', 'fjmass'], limits=(-100, 100))
    tf_dataResidualW_params = tf_dataResidualW(recoilscaled, msdscaled)
    tf_dataResidualZ = rl.BernsteinPoly("tf_dataResidualZ"+year, (1, 1), ['recoil', 'fjmass'], limits=(-100, 100))
    tf_dataResidualZ_params = tf_dataResidualZ(recoilscaled, msdscaled)
    #tf_paramsZ = rhalphabeth2D("Z+jets", tf_dataResidual_params, 3, 3)
    #tf_paramsW = rhalphabeth2D("W+jets", tf_dataResidual_params, 3, 2)

    model_dict = {}
    for recoilbin in range(nrecoil):

        sr_zjetsMCFailTemplate = template(background, "Z+jets", "nominal", recoilbin, "sr", "fail", read_sumw2=True)
        sr_zjetsMCFail = rl.TemplateSample(
            "sr" + year + "fail" + "recoil" + str(recoilbin) + "_zjetsMC",
            rl.Sample.BACKGROUND,
            sr_zjetsMCFailTemplate
        )
        sr_zjetsMCFail.setParamEffect(lumi, 1.027)
        sr_zjetsMCFail.setParamEffect(zjets_norm, 1.4)
        sr_zjetsMCFail.setParamEffect(trig_met, 1.02)
        sr_zjetsMCFail.setParamEffect(veto_tau, 1.03)
        sr_zjetsMCFail.setParamEffect(jec, 1.05)
        #sr_zjetsMCFail.autoMCStats()
        addBtagSyst(background, recoilbin, "Z+jets", "sr", sr_zjetsMCFail, "fail")
        addVJetsSyst(background, recoilbin, "Z+jets", "sr", sr_zjetsMCFail, "fail")

        sr_zhfMCFailTemplate = template(background, "Z+HF", "nominal", recoilbin, "sr", "fail", read_sumw2=True)
        sr_zhfMCFail = rl.TemplateSample(
            "sr" + year + "fail" + "recoil" + str(recoilbin) + "_zhfMC",
            rl.Sample.BACKGROUND,
            sr_zhfMCFailTemplate
        )
        sr_zhfMCFail.setParamEffect(zhf_fraction, 1.4)
        #sr_zhfMCFail.autoMCStats()

        sr_zlfMCFailTemplate = template(background, "Z+LF", "nominal", recoilbin, "sr", "fail", read_sumw2=True)
        sr_zlfMCFail = rl.TemplateSample(
            "sr" + year + "fail" + "recoil" + str(recoilbin) + "_zlfMC",
            rl.Sample.BACKGROUND,
            sr_zlfMCFailTemplate
        )
        sr_zlfMCFail.setParamEffect(zhf_fraction, 0.95)
        #sr_zlfMCFail.autoMCStats()

        sr_zjetsObservable = rl.Observable("fjmass", sr_zjetsMCFailTemplate[1])
        sr_zjetsParameters = np.array(
            [
                rl.IndependentParameter(
                    "sr" + year + "_zjets_fail_recoil"+str(recoilbin)+"_mass%d" % i,
                    0
                )
                for i in range(sr_zjetsObservable.nbins)
            ]
        )
        sr_zjetsBinYields = sr_zjetsMCFailTemplate[0] * (1 + (10./np.maximum(1., np.sqrt(sr_zjetsMCFailTemplate[0]))))**sr_zjetsParameters

        sr_zjetsFail = rl.ParametericSample(
            "sr" + year + "fail" + "recoil" + str(recoilbin)+ "_zjets",
            rl.Sample.BACKGROUND,
            sr_zjetsObservable,
            sr_zjetsBinYields
        )

        sr_wjetsMCFailTemplate = template(background, "W+jets", "nominal", recoilbin, "sr", "fail", read_sumw2=True)
        sr_wjetsMCFail = rl.TemplateSample(
            "sr" + year + "fail" + "recoil" + str(recoilbin) + "_wjetsMC",
            rl.Sample.BACKGROUND,
            sr_wjetsMCFailTemplate
        )
        sr_wjetsMCFail.setParamEffect(lumi, 1.027)
        sr_wjetsMCFail.setParamEffect(wjets_norm, 1.4)
        sr_wjetsMCFail.setParamEffect(trig_met, 1.02)
        sr_wjetsMCFail.setParamEffect(veto_tau, 1.03)
        sr_wjetsMCFail.setParamEffect(jec, 1.05)
        #sr_wjetsMCFail.autoMCStats()
        addBtagSyst(background, recoilbin, "W+jets", "sr", sr_wjetsMCFail, "fail")
        addVJetsSyst(background, recoilbin, "W+jets", "sr", sr_wjetsMCFail, "fail")

        sr_whfMCFailTemplate = template(background, "W+HF", "nominal", recoilbin, "sr", "fail", read_sumw2=True)
        sr_whfMCFail = rl.TemplateSample(
            "sr" + year + "fail" + "recoil" + str(recoilbin) + "_whfMC",
            rl.Sample.BACKGROUND,
            sr_whfMCFailTemplate
        )
        sr_whfMCFail.setParamEffect(whf_fraction, 1.4)
        #sr_whfMCFail.autoMCStats()

        sr_wlfMCFailTemplate = template(background, "W+LF", "nominal", recoilbin, "sr", "fail", read_sumw2=True)
        sr_wlfMCFail = rl.TemplateSample(
            "sr" + year + "fail" + "recoil" + str(recoilbin) + "_wlfMC",
            rl.Sample.BACKGROUND,
            sr_wlfMCFailTemplate
        )
        sr_wlfMCFail.setParamEffect(whf_fraction, 0.9)
        #sr_wlfMCFail.autoMCStats()

        sr_wjetsFailTransferFactor = sr_wjetsMCFail.getExpectation() / sr_zjetsMCFail.getExpectation()
        sr_wjetsFail = rl.TransferFactorSample(
            "sr" + year + "fail" + "recoil" + str(recoilbin)+ "_wjets",
            rl.Sample.BACKGROUND,
            sr_wjetsFailTransferFactor,
            sr_zjetsFail
        )

        sr_zjetsMCPassTemplate = template(background, "Z+jets", "nominal", recoilbin, "sr", "pass", read_sumw2=True)
        sr_zjetsMCPass = rl.TemplateSample(
            "sr" + year + "pass" + "recoil" + str(recoilbin) + "_zjetsMC",
            rl.Sample.BACKGROUND,
            sr_zjetsMCPassTemplate
        )
        sr_zjetsMCPass.setParamEffect(lumi, 1.027)
        sr_zjetsMCPass.setParamEffect(zjets_norm, 1.4)
        sr_zjetsMCPass.setParamEffect(trig_met, 1.02)
        sr_zjetsMCPass.setParamEffect(veto_tau, 1.03)
        sr_zjetsMCPass.setParamEffect(jec, 1.05)
        #sr_zjetsMCPass.autoMCStats()
        addBtagSyst(background, recoilbin, "Z+jets", "sr", sr_zjetsMCPass, "pass")
        addVJetsSyst(background, recoilbin, "Z+jets", "sr", sr_zjetsMCPass, "pass")

        sr_zhfMCPassTemplate = template(background, "Z+HF", "nominal", recoilbin, "sr", "pass", read_sumw2=True)
        sr_zhfMCPass = rl.TemplateSample(
            "sr" + year + "pass" + "recoil" + str(recoilbin) + "_zhfMC",
            rl.Sample.BACKGROUND,
            sr_zhfMCPassTemplate
        )
        sr_zhfMCPass.setParamEffect(zhf_fraction, 1.4)
        #sr_zhfMCPass.autoMCStats()

        sr_zlfMCPassTemplate = template(background, "Z+LF", "nominal", recoilbin, "sr", "pass", read_sumw2=True)
        sr_zlfMCPass = rl.TemplateSample(
            "sr" + year + "pass" + "recoil" + str(recoilbin) + "_zlfMC",
            rl.Sample.BACKGROUND,
            sr_zlfMCPassTemplate
        )
        sr_zlfMCPass.setParamEffect(zhf_fraction, 0.95)
        #sr_zlfMCPass.autoMCStats()

        tf_paramsZdeco = (sr_zlfMCPass.getExpectation()+sr_zhfMCPass.getExpectation()) / (sr_zlfMCFail.getExpectation()+sr_zhfMCFail.getExpectation())
        tf_paramsZ = tf_paramsZdeco * tf_dataResidualZ_params[recoilbin, :]

        sr_zjetsPass = rl.TransferFactorSample(
            "sr" + year + "pass" + "recoil" + str(recoilbin)+ "_zjets",
            rl.Sample.BACKGROUND,
            tf_paramsZ,
            sr_zjetsFail
        )

        sr_wjetsMCPassTemplate = template(background, "W+jets", "nominal", recoilbin, "sr", "pass", read_sumw2=True)
        sr_wjetsMCPass = rl.TemplateSample(
            "sr" + year + "pass" + "recoil" + str(recoilbin) + "_wjetsMC",
            rl.Sample.BACKGROUND,
            sr_wjetsMCPassTemplate
        )
        sr_wjetsMCPass.setParamEffect(lumi, 1.027)
        sr_wjetsMCPass.setParamEffect(wjets_norm, 1.4)
        sr_wjetsMCPass.setParamEffect(trig_met, 1.02)
        sr_wjetsMCPass.setParamEffect(veto_tau, 1.03)
        sr_wjetsMCPass.setParamEffect(jec, 1.05)
        #sr_wjetsMCPass.autoMCStats()
        addBtagSyst(background, recoilbin, "W+jets", "sr", sr_wjetsMCPass, "pass")
        addVJetsSyst(background, recoilbin, "W+jets", "sr", sr_wjetsMCPass, "pass")

        sr_whfMCPassTemplate = template(background, "W+HF", "nominal", recoilbin, "sr", "pass", read_sumw2=True)
        sr_whfMCPass = rl.TemplateSample(
            "sr" + year + "pass" + "recoil" + str(recoilbin) + "_whfMC",
            rl.Sample.BACKGROUND,
            sr_whfMCPassTemplate
        )
        sr_whfMCPass.setParamEffect(whf_fraction, 1.4)
        #sr_whfMCPass.autoMCStats()

        sr_wlfMCPassTemplate = template(background, "W+LF", "nominal", recoilbin, "sr", "pass", read_sumw2=True)
        sr_wlfMCPass = rl.TemplateSample(
            "sr" + year + "pass" + "recoil" + str(recoilbin) + "_wlfMC",
            rl.Sample.BACKGROUND,
            sr_wlfMCPassTemplate
        )
        sr_wlfMCPass.setParamEffect(whf_fraction, 0.9)
        #sr_wlfMCPass.autoMCStats()

        tf_paramsWdeco = (sr_wlfMCPass.getExpectation()+sr_whfMCPass.getExpectation()) / (sr_wlfMCFail.getExpectation()+sr_whfMCFail.getExpectation())
        tf_paramsW = tf_paramsWdeco * tf_dataResidualW_params[recoilbin, :]

        sr_wjetsPass = rl.TransferFactorSample(
            "sr" + year + "pass" + "recoil" + str(recoilbin)+ "_wjets",
            rl.Sample.BACKGROUND,
            tf_paramsW,
            sr_wjetsFail
        )

        for s in signal["sr"].identifiers("process"):
            #if "Mz500_mhs90_Mdm250" not in str(s):
            #    continue
            print("Signal is:", str(s))
            for category in ["pass", "fail"]:
                qcdpho_norm = rl.NuisanceParameter("qcdpho_norm" + year + category, "lnN")
                qcde_norm = rl.NuisanceParameter("qcde_norm" + year + category, "lnN")
                qcdmu_norm = rl.NuisanceParameter("qcdmu_norm" + year + category, "lnN")
                qcdsig_norm = rl.NuisanceParameter("qcdsig_norm" + year + category, "lnN")
                st_norm = rl.NuisanceParameter("st_norm" + year + category, "lnN")
                ttMC_norm = rl.NuisanceParameter("tt_norm" + year + category, "lnN")
                vv_norm = rl.NuisanceParameter("vv_norm" + year + category, "lnN")
                hbb_norm = rl.NuisanceParameter("hbb_norm" + year + category, "lnN")
                wjetsMC_norm = rl.NuisanceParameter("wjets_norm" + year + category, "lnN")
                zjetsMC_norm = rl.NuisanceParameter("zjets_norm" + year + category, "lnN")

                with open(
                        "data/"
                        + str(s).replace('_','')
                        + "-"
                        + year
                        + "-"
                        + category
                        + "-recoil"
                        + str(recoilbin)
                        + ".model",
                        "wb",
                ) as fout:
                    pickle.dump(model(year, recoilbin, category, s), fout, protocol=2)
